## Clone GitHub Repository

In [ ]:
# !git clone https://github.com/rjaditya-2702/alignment-project

In [1]:
from google.colab import userdata

In [ ]:
!pip install --upgrade setuptools wheel
# Attempting to install requirements while allowing a newer version of pomegranate if needed
!pip install datasets==2.14.0 # Using a more standard datasets version for Colab
!pip install pomegranate

In [ ]:
# Installing modern compatible versions to resolve version conflicts
!pip install datasets pomegranate dowhy scikit-learn statsmodels --upgrade

# Filter out problematic version-pinned requirements
!grep -vE 'pomegranate|datasets|dowhy|scikit-learn|statsmodels|scipy' alignment-project/requirements.txt > simple_requirements.txt
!pip install -r simple_requirements.txt

# Mount GDrive

In [3]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
source_dir = '/content/drive/MyDrive/Deep Learning/alignment-project'
destination_dir = '/content/alignment'

# Copy folder from Drive to local
try:
    if os.path.exists(source_dir):
        shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
        print(f'Successfully copied {source_dir} to {destination_dir}')
    else:
        print(f'Source directory in Drive {source_dir} not found.')
except Exception as e:
    print(f'Error occurred: {e}')

Mounted at /content/drive
Successfully copied /content/drive/MyDrive/Deep Learning/alignment-project to /content/alignment


In [ ]:
# Re-evaluating requirements.txt to install all other dependencies
import os

req_path = "alignment/requirements.txt"
if os.path.exists(req_path):
    with open(req_path, "r") as f:
        lines = f.readlines()

    # We only exclude the specific versions that we know fail on build or aren't in PyPI for 3.12
    # but we install the packages themselves without the problematic version constraint
    to_exclude = ['pomegranate', 'datasets', 'dowhy', 'scikit-learn', 'statsmodels', 'scipy', 'numpy', 'pandas']

    filtered_reqs = []
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        # If the package is in our exclude list, we skip the pinned version line
        if any(pkg in line.lower() for pkg in to_exclude):
            continue
        filtered_reqs.append(line)

    with open("full_requirements.txt", "w") as f:
        f.write("\n".join(filtered_reqs))

    print("Installing all remaining packages from requirements.txt...")
    !pip install -r full_requirements.txt
else:
    print("requirements.txt not found in alignment-project/")

Installing all remaining packages from requirements.txt...
  Using cached openai-2.32.0-py3-none-any.whl.metadata (31 kB)
Using cached openai-2.32.0-py3-none-any.whl (1.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 15.3 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.9.0
    Uninstalling jiter-0.9.0:
      Successfully uninstalled jiter-0.9.0
  Attempting uninstall: openai
    Found existing installation: openai 1.109.1
    Uninstalling openai-1.109.1:
      Successfully uninstalled openai-1.109.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.9 requires openai<2.0.0,>=1.66.3, but you have openai 2.32.0 which is incompatible.


---
# Below is a copy of train.ipynb
---

## Build data

Runs the full data pipeline in order. Each script skips work that is already done (checkpoint files).

| Script | Output | What it does |
|---|---|---|
| `src/data/build_dataset.py` | `dataset/unified.jsonl` | CLadder HF + CLadder synthetic + CauSciBench existing + CauSciBench synthetic |
| `src/data/split_dataset.py` | `dataset/train.jsonl`, `dataset/test.jsonl` | Synthetic → train, original benchmarks → test |
| `src/data/preprocess.py` | `output/train.jsonl`, `output/test.jsonl` | Rebuild prompts, normalize labels, add metadata |

**Skip this section** if `output/train.jsonl` and `output/test.jsonl` already exist.

In [4]:
import json
import sys
import random
from pathlib import Path
from collections import Counter

# Pointing ROOT to the cloned repository directory
ROOT = Path("/content/alignment").resolve()
sys.path.insert(0, str(ROOT))

print("Root set to:", ROOT)
# Verify src exists in the new root
if (ROOT / "src").exists():
    print("Found 'src' directory. Imports should now work.")
else:
    print("Warning: 'src' directory not found at", ROOT)

Root set to: /content/alignment
Found 'src' directory. Imports should now work.


In [5]:
import subprocess

# Check what already exists
to_check = {
    "dataset/unified.jsonl": ROOT / "dataset" / "unified.jsonl",
    "dataset/train.jsonl":   ROOT / "dataset" / "train.jsonl",
    "dataset/test.jsonl":    ROOT / "dataset" / "test.jsonl",
    "output/train.jsonl":    ROOT / "output"  / "train.jsonl",
    "output/test.jsonl":     ROOT / "output"  / "test.jsonl",
}
for label, path in to_check.items():
    status = f"{sum(1 for _ in open(path)):,} rows" if path.exists() else "MISSING"
    print(f"  {'✓' if path.exists() else '✗'}  {label:<30}  {status}")

  ✓  dataset/unified.jsonl           96,425 rows
  ✓  dataset/train.jsonl             87,194 rows
  ✓  dataset/test.jsonl              9,231 rows
  ✓  output/train.jsonl              87,194 rows
  ✓  output/test.jsonl               9,231 rows


In [5]:
# !rm alignment/dataset/ckpt_*.jsonl
# !rm alignment/dataset/ckpt_4_causci_synth.jsonl

In [6]:
# !pip install -r alignment/original_data/CauSciBench/requirements.txt
# !pip install -r alignment/original_data/Cladder/requirements.txt

In [7]:
# !pip install omnibelt==0.7.6 omniply==0.1.1 omnifig==1.0.1

In [8]:
# from alignment.src.data.build_dataset import build

# UNIFIED = ROOT / "dataset" / "unified.jsonl"

# if UNIFIED.exists():
#     print(f"unified.jsonl exists ({sum(1 for _ in open(UNIFIED)):,} rows) — skipping build_dataset.py")
# else:
#     print("Running build_dataset.py  (this takes a while — CLadder synthetic ~101k rows, CauSciBench uses OpenAI API)")
#     build()

In [9]:
# TRAIN_DS = ROOT / "dataset" / "train.jsonl"
# TEST_DS  = ROOT / "dataset" / "test.jsonl"

# if TRAIN_DS.exists() and TEST_DS.exists():
#     print(f"dataset/train.jsonl ({sum(1 for _ in open(TRAIN_DS)):,} rows) and dataset/test.jsonl ({sum(1 for _ in open(TEST_DS)):,} rows) exist — skipping split_dataset.py")
# else:
#     print("Running split_dataset.py ...")
#     result = subprocess.run(
#         [sys.executable, str(ROOT / "src" / "data" / "split_dataset.py")],
#         cwd=str(ROOT),
#     )
#     print(f"\nExit code: {result.returncode}")

In [6]:
import json
from collections import Counter

# load training jsonl from dataset/
with open(ROOT / "dataset" / "train.jsonl") as f:
    train_rows = [json.loads(l) for l in f]

print(Counter(r["source"] for r in train_rows))

Counter({'cladder_synthetic': 86744, 'causcibench_synthetic': 450})


In [14]:
# !rm -r /content/alignment/src

In [7]:
from alignment.src.data.preprocess import preprocess

In [8]:
OUTPUT_TRAIN = ROOT / "output" / "train.jsonl"
OUTPUT_TEST  = ROOT / "output" / "test.jsonl"

preprocess()

Wrote 87194 train rows → /content/alignment/output/train.jsonl
Wrote 9231 test rows  → /content/alignment/output/test.jsonl

=== Row Counts ===
train: cladder=86744  causcibench=450
test:  cladder=8917  causcibench=314

=== CLadder Label Balance ===
train: yes=41452  no=45292
test: yes=4459  no=4458

=== CLadder NaN Reasoning ===
train: 112
test:  0
  NOTE: 112 train rows have empty step4 — kept as-is, step4 set to null

=== CauSciBench Methods in Train ===
ols=50  iv=50  did=50  rdd=50  matching=50  ipw=50  glm=50  frontdoor=50  diff_in_means=50

=== CauSciBench Methods in Test ===
ols=66  iv=79  did=56  rdd=46  matching=44  ipw=0  glm=7  frontdoor=15  diff_in_means=0
  WARN: missing methods: ['ipw', 'diff_in_means']

=== Prompt Length (whitespace-split word count) ===
cladder:   min=1137  mean=1169  max=1356
causcibench:   min=1470  mean=1728  max=6761
  WARN: 4 prompts exceed 5000 words: ['causci_real_44', 'causci_real_115', 'causci_real_116', 'causci_real_117']

=== CSV Load Failur

In [9]:
import json
from collections import Counter

# load training jsonl from dataset/
with open(ROOT / "output" / "train.jsonl") as f:
    train_rows = [json.loads(l) for l in f]

print(Counter(r["source"] for r in train_rows))

Counter({'cladder': 86744, 'causcibench': 450})


I commented the above cells because i already ran the code.

I have uploaded the final train and test splits in ./alignment-project/output/ folder

In [10]:
# Final status — all five files should show ✓ before proceeding
print("Data status:")
for label, path in to_check.items():
    status = f"{sum(1 for _ in open(path)):,} rows" if path.exists() else "MISSING"
    print(f"  {'✓' if path.exists() else '✗'}  {label:<30}  {status}")

Data status:
  ✓  dataset/unified.jsonl           96,425 rows
  ✓  dataset/train.jsonl             87,194 rows
  ✓  dataset/test.jsonl              9,231 rows
  ✓  output/train.jsonl              87,194 rows
  ✓  output/test.jsonl               9,231 rows


# 1. inspect training data

In [11]:
print(ROOT)
with open(ROOT / "output" / "train.jsonl") as f:
    train_rows = [json.loads(l) for l in f]

with open(ROOT / "output" / "test.jsonl") as f:
    test_rows = [json.loads(l) for l in f]

print(train_rows[-1])

print(f"Train: {len(train_rows):,}  |  Test: {len(test_rows):,}")
print()
print("Train sources:", dict(Counter(r["source"] for r in train_rows)))
print("Test  sources:", dict(Counter(r["source"] for r in test_rows)))

/content/alignment
{'id': 'causci_new_synth_glm_49', 'source': 'causcibench', 'split': 'train', 'prompt': 'You are given a dataset from a research study along with a description of how the data was collected. Your task is to estimate the effect of one variable on another by following these steps precisely.\n\n## Study Description\nThis dataset comes from an observational study of adult patients treated for an acute medical condition in a hospital setting. It includes demographic and clinical information recorded at the start of care, along with whether patients received a specialized treatment and whether they had recovered by the follow-up visit.\n\n## Dataset\nPath: dataset/synthetic_causci/glm_49.csv\nShape: 741 rows, 9 columns\n\nColumns and types:\n  patient_age: int64\n  baseline_severity_score: int64\n  body_mass_index: int64\n  days_since_symptom_onset: int64\n  has_diabetes: int64\n  has_hypertension: int64\n  has_smoking_history: int64\n  received_specialized_treatment: int64

In [12]:
# CLadder sample
cl = next(r for r in train_rows if r["source"] == "cladder")
print("=== CLadder sample ===")
print("id:", cl["id"])
print("label:", cl["label"])
print("query type:", cl["groundtruth"]["step2"])
print()
print("--- prompt (first 800 chars) ---")
print(cl["prompt"][:800])

=== CLadder sample ===
id: cladder_synth_alarm_mediation_0
label: yes
query type: ate

--- prompt (first 800 chars) ---
You are given a scenario describing relationships between variables, along with numerical data and a question. Your task is to determine the answer by following these steps precisely.
---

Strict rules (follow these exactly):
- Output ONLY the five numbered steps in order.
- Nothing before "## Step 1" and nothing after the single word in Step 5.
- Write each step exactly once.
- Each step must be short and direct. No long paragraphs or verbosity.
- Do not repeat content from previous steps.
- Step 5 must contain exactly one word on its own line: "Yes" or "No". No quotes, no extra text, no code, no periods.
- Do not repeat any step, any code block, or the word "Yes".
- Stop immediately after Step 5. Do not continue generating. No extra sentences, no "Okay let's see", no repetition.

### Que


In [13]:
# CauSciBench sample
cs = next(r for r in train_rows if r["source"] == "causcibench")
print("=== CauSciBench sample ===")
print("id:", cs["id"])
print("label:", cs["label"])
print("method:", cs["groundtruth"]["step2"])
print("step1 gt:", cs["groundtruth"]["step1"])
print()
print("--- prompt (first 800 chars) ---")
print(cs["prompt"][:800])

=== CauSciBench sample ===
id: causci_new_synth_diff_in_means_0
label: 5.117014013771599
method: diff_in_means
step1 gt: {'treatment': 'after_school_tutoring', 'outcome': 'endline_test_score', 'controls': ['baseline_math_score', 'days_absent_last_term', 'parental_education_level', 'attended_study_hall', 'has_tutoring_support', 'eligible_for_free_lunch'], 'instrument': None, 'running_variable': None, 'time_variable': None, 'group_variable': None, 'mediator': None}

--- prompt (first 800 chars) ---
You are given a dataset from a research study along with a description of how the data was collected. Your task is to estimate the effect of one variable on another by following these steps precisely.

## Study Description
This dataset comes from a randomized evaluation of an after-school tutoring program for middle school students. It includes baseline academic and household background information, along with indicators for school support services and student needs, to assess how the program 

In [14]:
# Label distributions
cl_train = [r for r in train_rows if r["source"] == "cladder"]
print("CLadder train labels:", dict(Counter(r["label"] for r in cl_train)))

cl_query_types = Counter(r["groundtruth"]["step2"] for r in cl_train)
print("CLadder query types:", dict(cl_query_types.most_common()))

cs_train = [r for r in train_rows if r["source"] == "causcibench"]
print("\nCauSciBench methods:", dict(Counter(r["groundtruth"]["step2"] for r in cs_train)))

CLadder train labels: {'yes': 41452, 'no': 45292}
CLadder query types: {'ate': 24348, 'ett': 21348, 'marginal': 13694, 'correlation': 13660, 'nie': 6882, 'nde': 3200, 'collider_bias': 2100, 'exp_away': 1400, 'backadj': 112}

CauSciBench methods: {'diff_in_means': 50, 'ols': 50, 'ipw': 50, 'matching': 50, 'iv': 50, 'did': 50, 'rdd': 50, 'frontdoor': 50, 'glm': 50}


# model loading

In [15]:
# Installing specific recent versions of transformers, peft, and huggingface_hub for compatibility
# !pip install --force-reinstall transformers peft huggingface_hub tokenizers --upgrade

In [16]:
!nvcc --version
# or


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [17]:
# # 1. Full clean uninstall
# !pip uninstall -y torch torchvision torchaudio nvidia-nccl-cu12 nvidia-cuda-runtime-cu12 nvidia-cudnn-cu12 nvidia-nccl-cu12

# # 2. Install latest compatible NCCL from NVIDIA first
# !pip install nvidia-nccl-cu12 --force-reinstall --no-deps

# # 3. Reinstall PyTorch for CUDA 12.8
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --force-reinstall

In [20]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [18]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  {props.total_memory / 1e9:.1f} GB")

CUDA available: True
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition  102.0 GB


In [35]:
# del model
# del tokenizer

In [21]:
from alignment.src.training.train import load_policy

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# MODEL_NAME = str(ROOT / "output" / "checkpoints" / "step_500")  # resume

model, tokenizer = load_policy(MODEL_NAME)
device = str(next(model.parameters()).device)
print("\nModel device:", device)

Loading tokenizer from Qwen/Qwen2.5-7B-Instruct
Loading model from Qwen/Qwen2.5-7B-Instruct → cuda


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491

Model device: cuda:0


In [22]:
# Memory check after loading
import torch
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        print(f"GPU {i}: {alloc:.1f} GB allocated  /  {reserved:.1f} GB reserved")

GPU 0: 30.9 GB allocated  /  31.1 GB reserved


# generation smoke test

In [23]:
from src.training.train import generate_rollouts

sample_row = cl_train[0]
print("Prompt (first 400 chars):")
print(sample_row["prompt"][:400])
print("\nGenerating 2 completions...")

completions = generate_rollouts(model, tokenizer, sample_row["prompt"], n=2, device=device)

for i, c in enumerate(completions):
    print(f"\n{'='*60}")
    print(f"Completion {i+1}:")
    print(c[:600])

Prompt (first 400 chars):
You are given a scenario describing relationships between variables, along with numerical data and a question. Your task is to determine the answer by following these steps precisely.
---

Strict rules (follow these exactly):
- Output ONLY the five numbered steps in order.
- Nothing before "## Step 1" and nothing after the single word in Step 5.
- Write each step exactly once.
- Each step must be 

Generating 2 completions...

Completion 1:
## Step 1: Causal Structure
V1 -> V2, where V1 is "Husband sets alarm" and V2 is "Alarm rings".

## Step 2: Query Classification
correlation

## Step 3: Derive Estimand
P(V2=1|V1=1) vs P(V2=1|V1=0). Given: P(V2=1|V1=1) = 0.79, P(V2=1|V1=0) = 0.43.

## Step 4: Compute
```python
P_V2_given_V1_1 = 0.79
P_V2_given_V1_0 = 0.43

if P_V2_given_V1_1 > P_V2_given_V1_0:
    result = 'yes'
else:
    result = 'no'

print(result)
```

## Step 5: Answer
Yes

Completion 2:
## Step 1: Causal Structure
V1 -> V2

## Step 2: Query Classificat

In [24]:
completions[0]


'## Step 1: Causal Structure\nV1 -> V2, where V1 is "Husband sets alarm" and V2 is "Alarm rings".\n\n## Step 2: Query Classification\ncorrelation\n\n## Step 3: Derive Estimand\nP(V2=1|V1=1) vs P(V2=1|V1=0). Given: P(V2=1|V1=1) = 0.79, P(V2=1|V1=0) = 0.43.\n\n## Step 4: Compute\n```python\nP_V2_given_V1_1 = 0.79\nP_V2_given_V1_0 = 0.43\n\nif P_V2_given_V1_1 > P_V2_given_V1_0:\n    result = \'yes\'\nelse:\n    result = \'no\'\n\nprint(result)\n```\n\n## Step 5: Answer\nYes'

In [25]:
# Score those completions
from alignment.src.training.reward import compute_rewards
rewards = compute_rewards(completions, [sample_row] * 2)
print("Rewards:", rewards)
print("GT query type:", sample_row["groundtruth"]["step2"])
print("GT answer:", sample_row["label"])

Rewards: [-269.0, -269.0]
GT query type: ate
GT answer: yes


## Training

Calls `train()` from `src/training/train.py` directly.

**Interrupt at any time** — checkpoints are saved every `SAVE_EVERY` steps and after each epoch.  
**Resume** by setting `RESUME_FROM` to the checkpoint path.

Default hyperparameters:
| Param | Value |
|---|---|
| N rollouts | 8 |
| LoRA r | 16 |
| β (KL) | 0.01 |
| LR | 2e-5 |
| Grad accum | 8 prompts |
| Epochs | 3 |

In [26]:
import argparse
from src.training.train import (
    BETA, GRAD_ACCUM, LOG_EVERY, MAX_EPOCHS,
    N_ROLLOUTS, OUTPUT_DIR, SANDBOX_WORKERS, SAVE_EVERY,
    LR,
)

RESUME_FROM = None  # e.g. str(ROOT / "output" / "checkpoints" / "step_500")

args = argparse.Namespace(
    model       = MODEL_NAME,
    resume      = RESUME_FROM,
    output_dir  = str(ROOT / "output" / "checkpoints"),
    epochs      = MAX_EPOCHS,
    n_rollouts  = N_ROLLOUTS,
    beta        = BETA,
    lr          = LR,
    grad_accum  = GRAD_ACCUM,
    save_every  = SAVE_EVERY,
    log_every   = LOG_EVERY,
    sandbox_workers = SANDBOX_WORKERS,
)
print("Training config:")
for k, v in vars(args).items():
    print(f"  {k}: {v}")

Training config:
  model: Qwen/Qwen2.5-7B-Instruct
  resume: None
  output_dir: /content/alignment/output/checkpoints
  epochs: 3
  n_rollouts: 8
  beta: 0.01
  lr: 2e-05
  grad_accum: 8
  save_every: 500
  log_every: 10
  sandbox_workers: 8


In [ ]:
# The model is already loaded above — pass it directly to avoid reloading.
# This replicates train() but reuses the model from cell 3.

import random
import torch
import torch.nn.functional as F
from src.training.train import generate_rollouts, sequence_logprob, grpo_loss, MAX_PROMPT_LEN, MAX_NEW_TOKENS
from src.training.reward import compute_rewards

out_dir = Path(args.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=args.lr,
    weight_decay=0.01,
)

global_step = 0

for epoch in range(args.epochs):
    random.shuffle(train_rows)
    optimizer.zero_grad()
    accum_loss = accum_reward = 0.0
    n_accum = 0

    for row in train_rows:
        # 1. Generate rollouts
        model.eval()
        completions = generate_rollouts(model, tokenizer, row["prompt"], args.n_rollouts, device)
        model.train()

        # 2. Score
        rewards_list = compute_rewards(completions, [row] * args.n_rollouts, args.sandbox_workers)
        rewards = torch.tensor(rewards_list, dtype=torch.float32, device=device)

        if rewards.std() < 1e-6:
            continue

        # 3. Tokenize
        prompt_ids = tokenizer(
            row["prompt"], return_tensors="pt", truncation=True,
            max_length=MAX_PROMPT_LEN, add_special_tokens=True,
        ).input_ids[0].to(device)

        policy_lps, ref_lps = [], []
        for completion in completions:
            comp_ids = tokenizer(
                completion, return_tensors="pt", add_special_tokens=False,
                truncation=True, max_length=MAX_NEW_TOKENS,
            ).input_ids[0].to(device)

            if comp_ids.shape[0] == 0:
                z = torch.zeros(1, device=device)
                policy_lps.append(z.squeeze().requires_grad_(True))
                ref_lps.append(z.squeeze())
                continue

            lp = sequence_logprob(model, prompt_ids, comp_ids)
            policy_lps.append(lp)

            model.disable_adapter_layers()
            with torch.no_grad():
                ref_lp = sequence_logprob(model, prompt_ids, comp_ids)
            model.enable_adapter_layers()
            ref_lps.append(ref_lp)

        # 4. GRPO loss
        loss = grpo_loss(
            torch.stack(policy_lps),
            torch.stack(ref_lps).detach(),
            rewards,
            beta=args.beta,
        )
        (loss / args.grad_accum).backward()

        accum_loss   += loss.item()
        accum_reward += rewards.mean().item()
        n_accum      += 1
        global_step  += 1

        # 5. Optimizer step
        if global_step % args.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        # Log
        if global_step % args.log_every == 0 and n_accum > 0:
            print(
                f"epoch={epoch+1}  step={global_step:>6}  "
                f"loss={accum_loss/n_accum:.4f}  "
                f"reward={accum_reward/n_accum:.3f}",
                flush=True,
            )
            accum_loss = accum_reward = 0.0
            n_accum = 0

        # Checkpoint
        if global_step % args.save_every == 0:
            ckpt = out_dir / f"step_{global_step}"
            model.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)
            print(f"Saved → {ckpt}")

    ckpt = out_dir / f"epoch_{epoch+1}"
    model.save_pretrained(ckpt)
    tokenizer.save_pretrained(ckpt)
    print(f"Epoch {epoch+1} complete → {ckpt}")

final = out_dir / "final"
model.save_pretrained(final)
tokenizer.save_pretrained(final)
print(f"Training complete → {final}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


epoch=1  step=    10  loss=0.0063  reward=-274.600
epoch=1  step=    20  loss=-0.0025  reward=-194.062
epoch=1  step=    30  loss=-0.0073  reward=-154.500
epoch=1  step=    40  loss=-0.0052  reward=-181.400
epoch=1  step=    50  loss=-0.0107  reward=-217.975
epoch=1  step=    60  loss=-0.0137  reward=-105.037
epoch=1  step=    70  loss=-0.0027  reward=-97.812
epoch=1  step=    80  loss=0.0012  reward=-103.987
epoch=1  step=    90  loss=-0.0015  reward=-115.775
epoch=1  step=   100  loss=-0.0087  reward=-103.713
epoch=1  step=   110  loss=-0.0012  reward=-151.500
epoch=1  step=   120  loss=0.0006  reward=-89.963
epoch=1  step=   130  loss=0.0034  reward=-127.675
epoch=1  step=   140  loss=0.0061  reward=-55.900
epoch=1  step=   150  loss=0.0010  reward=-96.150
epoch=1  step=   160  loss=0.0014  reward=-178.375
epoch=1  step=   170  loss=-0.0011  reward=-127.200
epoch=1  step=   180  loss=-0.0004  reward=-75.375
epoch=1  step=   190  loss=-0.0017  reward=-91.525
epoch=1  step=   200  los

# eval

In [ ]:
# Run from terminal (eval loads its own model copy to avoid state contamination):
#
#   python src/eval/eval.py --model output/checkpoints/final --output-dir output/eval_post_grpo
#
# Or inline — reuse the already-loaded model:

from src.eval.eval import run_eval
from src.eval.metrics import aggregate_metrics
import json

# Limit to first 200 rows for a quick check; remove limit for full eval
EVAL_LIMIT = 200

model.eval()
results = run_eval(
    test_rows[:EVAL_LIMIT],
    model,
    tokenizer,
    use_llm_judge=False,   # set True if you have OPENAI_API_KEY set
    sandbox_workers=8,
)

metrics = aggregate_metrics(results)

if "cladder" in metrics:
    cl = metrics["cladder"]
    print(f"CLadder (n={cl['n']})")
    print(f"  Accuracy: {cl['accuracy']:.1f}%   Avg score: {cl['avg_score']:.1f}/100")

if "causcibench" in metrics:
    cs = metrics["causcibench"]
    print(f"CauSciBench (n={cs['n']})")
    print(f"  Method acc: {cs['method_accuracy']:.1f}%   Code rate: {cs['code_execution_rate']:.1f}%   Avg score: {cs['avg_score']:.1f}/100")

In [ ]:
#save model to drive


In [ ]:
import os
from google.colab import drive

# Mount Google Drive if it's not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define the path in Google Drive where you want to save the model
drive_save_path = '/content/drive/MyDrive/alignment_model_checkpoints/final_trained_model'

# Create the directory if it doesn't exist
os.makedirs(drive_save_path, exist_ok=True)

print(f"Saving model to: {drive_save_path}")

# Save the model and tokenizer
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print("Model and tokenizer saved successfully to Google Drive!")

# *fin.*